# MediSense AI — NIH ChestX-ray14
## DenseNet-121 vs DenseNet-121 + CBAM — Kaggle GPU Fast Experiment

**Kaggle-specific version.** Attach the NIH ChestX-ray14 dataset and select a GPU accelerator.

### This version is intentionally fast
- 30,000 training images
- 3,000 validation images
- 5,000 development-test images
- 224×224 images
- Batch size 64 (use 32 if GPU OOM)
- 8 data-loader workers when available
- AMP using the current PyTorch API
- Early stopping
- DenseNet-121 and DenseNet-121 + CBAM only
- 14-label multi-label classification
- Validation-derived F1 thresholds (no test-threshold leakage)

**Important:** this is for MediSense model selection and efficient experimentation. The 5,000-image development test is not a definitive full NIH ChestX-ray14 benchmark.


## Kaggle setup — do this before Run All

1. Open this notebook in **Kaggle**.
2. Attach your NIH ChestX-ray14 dataset with **Add Input**.
3. Go to **Settings → Accelerator → GPU**.
4. Run **all cells from the beginning**.
5. If you see `Device: cuda` and your GPU name, the setup is correct.
6. Do not change dataset paths manually — the notebook searches `/kaggle/input` automatically.
7. If you get CUDA out-of-memory, stop the run and change only `BATCH_SIZE = 64` to `32`.

The output is written to `/kaggle/working/medisense_chest_densenet_cbam_fast/`.


In [2]:
# 1. KAGGLE GPU CONFIGURATION
# ------------------------------------------------------------
# This notebook is specifically for Kaggle.
# Attach the NIH ChestX-ray14 dataset and enable a GPU.
#
# No /content paths and no Google Drive paths are used.

from pathlib import Path
import os, json, random, time, copy, gc

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Torchvision:", __import__("torchvision").__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    print("WARNING: GPU is not enabled. In Kaggle: Settings -> Accelerator -> GPU")

# ------------------------------------------------------------
# SPEED / EXPERIMENT SETTINGS
# ------------------------------------------------------------

TRAIN_LIMIT = 30000
VAL_LIMIT = 3000
TEST_LIMIT = 5000

IMAGE_SIZE = 224

# 64 is faster if your Kaggle GPU has enough VRAM.
# If you get CUDA out-of-memory, change only this to 32.
BATCH_SIZE = 64

# Kaggle CPU workers. 8 is usually a good starting point for JPEG-heavy loading.
NUM_WORKERS = min(8, os.cpu_count() or 2)

HEAD_EPOCHS = 2
FINETUNE_EPOCHS = 5
PATIENCE = 2

LR_HEAD = 3e-4
LR_FINETUNE = 1e-5
WEIGHT_DECAY = 1e-4

NUM_CLASSES = 14

OUTPUT_DIR = Path("/kaggle/working/medisense_chest_densenet_cbam_fast")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_AMP = torch.cuda.is_available()

# Performance settings
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

print()
print("Train images:", TRAIN_LIMIT)
print("Validation images:", VAL_LIMIT)
print("Development-test images:", TEST_LIMIT)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Workers:", NUM_WORKERS)
print("AMP:", USE_AMP)
print("Output:", OUTPUT_DIR)


PyTorch: 2.10.0+cu128
Torchvision: 0.25.0+cu128
Device: cuda
GPU: Tesla T4
CUDA: 12.8

Train images: 30000
Validation images: 3000
Development-test images: 5000
Image size: 224
Batch size: 64
Workers: 4
AMP: True
Output: /kaggle/working/medisense_chest_densenet_cbam_fast


In [3]:
# 2. FIND NIH CHESTX-RAY14 DATASET ON KAGGLE

CSV_NAME = "Data_Entry_2017.csv"
TRAIN_VAL_NAME = "train_val_list.txt"
TEST_NAME = "test_list.txt"

INPUT_ROOT = Path("/kaggle/input")

candidates = list(INPUT_ROOT.rglob(CSV_NAME))

if not candidates:
    raise FileNotFoundError(
        "Data_Entry_2017.csv was not found under /kaggle/input.\n"
        "Attach your NIH ChestX-ray14 dataset to this Kaggle notebook."
    )

scored = []

for csv_path in candidates:
    root = csv_path.parent
    score = 0

    if (root / TRAIN_VAL_NAME).exists():
        score += 5
    if (root / TEST_NAME).exists():
        score += 5
    if any(root.glob("images_*")):
        score += 5

    scored.append((score, root))

DATA_ROOT = max(scored, key=lambda x: x[0])[1]

CSV_PATH = DATA_ROOT / CSV_NAME
TRAIN_VAL_PATH = DATA_ROOT / TRAIN_VAL_NAME
TEST_PATH = DATA_ROOT / TEST_NAME

IMAGE_DIRS = sorted(
    p for p in DATA_ROOT.glob("images_*")
    if p.is_dir()
)

print("Dataset root:", DATA_ROOT)
print("CSV:", CSV_PATH)
print("Train/val list:", TRAIN_VAL_PATH)
print("Test list:", TEST_PATH)
print("Image folders:", len(IMAGE_DIRS))

for p in IMAGE_DIRS:
    nested = p / "images"
    print(" ", p.name, "->", "nested images/" if nested.exists() else "direct")

assert CSV_PATH.exists()
assert TRAIN_VAL_PATH.exists()
assert TEST_PATH.exists()
assert IMAGE_DIRS, "No images_* folders found."


Dataset root: /kaggle/input/datasets/nih-chest-xrays/data
CSV: /kaggle/input/datasets/nih-chest-xrays/data/Data_Entry_2017.csv
Train/val list: /kaggle/input/datasets/nih-chest-xrays/data/train_val_list.txt
Test list: /kaggle/input/datasets/nih-chest-xrays/data/test_list.txt
Image folders: 12
  images_001 -> nested images/
  images_002 -> nested images/
  images_003 -> nested images/
  images_004 -> nested images/
  images_005 -> nested images/
  images_006 -> nested images/
  images_007 -> nested images/
  images_008 -> nested images/
  images_009 -> nested images/
  images_010 -> nested images/
  images_011 -> nested images/
  images_012 -> nested images/


In [4]:
# 3. BUILD / REUSE IMAGE INDEX
# Handles:
#   /kaggle/input/.../images_001/*.png
# and
#   /kaggle/input/.../images_001/images/*.png

INDEX_PATH = OUTPUT_DIR / "image_index.csv"

if INDEX_PATH.exists():
    index_df = pd.read_csv(INDEX_PATH)
    if not {"filename", "path"}.issubset(index_df.columns):
        INDEX_PATH.unlink()
        index_df = None
else:
    index_df = None

if index_df is None:
    records = []

    for image_dir in IMAGE_DIRS:
        search_root = image_dir / "images"
        if not search_root.exists():
            search_root = image_dir

        count = 0

        for p in search_root.rglob("*"):
            if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg"}:
                records.append((p.name, str(p)))
                count += 1

        print(f"{image_dir.name}: {count:,} images")

    index_df = (
        pd.DataFrame(records, columns=["filename", "path"])
        .drop_duplicates("filename")
        .reset_index(drop=True)
    )

    index_df.to_csv(INDEX_PATH, index=False)

image_index = dict(zip(index_df["filename"], index_df["path"]))

print("Indexed unique images:", f"{len(image_index):,}")
assert len(image_index) > 100000, "Unexpectedly few images indexed."


Indexed unique images: 112,120


In [5]:
# 4. LOAD METADATA + 14 LABELS

df = pd.read_csv(CSV_PATH)

PATHOLOGIES = [
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pneumonia",
    "Pneumothorax",
    "Consolidation",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Pleural_Thickening",
    "Hernia"
]

def make_multihot(label_string):
    labels = {
        x.strip()
        for x in str(label_string).split("|")
    }

    return [
        int(pathology in labels)
        for pathology in PATHOLOGIES
    ]

targets = np.asarray(
    df["Finding Labels"]
    .apply(make_multihot)
    .tolist(),
    dtype=np.float32
)

for i, pathology in enumerate(PATHOLOGIES):
    df[pathology] = targets[:, i]

df["image_path"] = (
    df["Image Index"]
    .map(image_index)
)

df = df.dropna(
    subset=["image_path"]
).reset_index(drop=True)

print("Usable images:", f"{len(df):,}")
print("Patients:", f"{df['Patient ID'].nunique():,}")

assert len(df) > 100000


Usable images: 112,120
Patients: 30,805


In [6]:
# 5. OFFICIAL TEST + PATIENT-LEVEL VALIDATION SPLIT

def read_list(path):
    with open(path, "r") as f:
        return [
            line.strip()
            for line in f
            if line.strip()
        ]

train_val_names = set(
    read_list(TRAIN_VAL_PATH)
)

test_names = set(
    read_list(TEST_PATH)
)

train_val_df = df[
    df["Image Index"].isin(train_val_names)
].copy()

test_df = df[
    df["Image Index"].isin(test_names)
].copy()

patients = (
    train_val_df["Patient ID"]
    .drop_duplicates()
    .to_numpy()
)

train_patients, val_patients = train_test_split(
    patients,
    test_size=0.10,
    random_state=SEED
)

train_df = train_val_df[
    train_val_df["Patient ID"].isin(train_patients)
].copy()

val_df = train_val_df[
    train_val_df["Patient ID"].isin(val_patients)
].copy()

print("Official train/val:", f"{len(train_val_df):,}")
print("Official test:", f"{len(test_df):,}")
print()
print("Patient-level train:", f"{len(train_df):,}")
print("Patient-level val:", f"{len(val_df):,}")

assert set(train_df["Patient ID"]).isdisjoint(
    val_df["Patient ID"]
)

assert set(train_df["Patient ID"]).isdisjoint(
    test_df["Patient ID"]
)

assert set(val_df["Patient ID"]).isdisjoint(
    test_df["Patient ID"]
)


Official train/val: 86,524
Official test: 25,596

Patient-level train: 77,988
Patient-level val: 8,536


In [7]:
# 6. SELECT FAST EXPERIMENT SUBSETS

train_run_df = train_df.sample(
    min(TRAIN_LIMIT, len(train_df)),
    random_state=SEED
).copy()

val_run_df = val_df.sample(
    min(VAL_LIMIT, len(val_df)),
    random_state=SEED
).copy()

test_run_df = test_df.sample(
    min(TEST_LIMIT, len(test_df)),
    random_state=SEED
).copy()

print("FAST EXPERIMENT")
print("Train:", f"{len(train_run_df):,}")
print("Val:", f"{len(val_run_df):,}")
print("Test:", f"{len(test_run_df):,}")


FAST EXPERIMENT
Train: 30,000
Val: 3,000
Test: 5,000


In [8]:
# 7. DATASET + DATALOADERS

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class ChestXrayDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]

        with Image.open(row["image_path"]) as img:
            img = img.convert("RGB")
            image = self.transform(img)

        target = torch.tensor(
            row[PATHOLOGIES].to_numpy(dtype=np.float32),
            dtype=torch.float32
        )

        return image, target


train_ds = ChestXrayDataset(train_run_df, train_transform)
val_ds = ChestXrayDataset(val_run_df, eval_transform)
test_ds = ChestXrayDataset(test_run_df, eval_transform)

loader_args = dict(
    batch_size=BATCH_SIZE,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)

if NUM_WORKERS > 0:
    loader_args["num_workers"] = NUM_WORKERS
    loader_args["prefetch_factor"] = 2

train_loader = DataLoader(
    train_ds, shuffle=True, drop_last=True, **loader_args
)

val_loader = DataLoader(
    val_ds, shuffle=False, **loader_args
)

test_loader = DataLoader(
    test_ds, shuffle=False, **loader_args
)

x, y = next(iter(train_loader))

print("Batch images:", tuple(x.shape))
print("Batch labels:", tuple(y.shape))
print("Approx. train steps/epoch:", len(train_loader))


Batch images: (64, 3, 224, 224)
Batch labels: (64, 14)
Approx. train steps/epoch: 468


In [9]:
# 8. POSITIVE CLASS WEIGHTS

positive_counts = (
    train_run_df[PATHOLOGIES]
    .sum(axis=0)
    .to_numpy(dtype=np.float32)
)

negative_counts = (
    len(train_run_df) -
    positive_counts
)

pos_weight_np = (
    negative_counts /
    np.maximum(
        positive_counts,
        1.0
    )
)

POS_WEIGHT = torch.tensor(
    pos_weight_np,
    dtype=torch.float32,
    device=DEVICE
)

weights_df = pd.DataFrame({
    "Disease": PATHOLOGIES,
    "Positive": positive_counts.astype(int),
    "Negative": negative_counts.astype(int),
    "PosWeight": pos_weight_np
})

display(
    weights_df.style.format({
        "PosWeight": "{:.2f}"
    })
)


,Disease,Positive,Negative,PosWeight
0,Atelectasis,2904,27096,9.33
1,Cardiomegaly,604,29396,48.67
2,Effusion,3014,26986,8.95
3,Infiltration,4886,25114,5.14
4,Mass,1465,28535,19.48
5,Nodule,1646,28354,17.23
6,Pneumonia,307,29693,96.72
7,Pneumothorax,927,29073,31.36
8,Consolidation,1029,28971,28.15
9,Edema,470,29530,62.83


In [10]:
# 9. DENSENET-121 BASELINE

def build_densenet121():

    model = models.densenet121(
        weights=models.DenseNet121_Weights.DEFAULT
    )

    model.classifier = nn.Linear(
        model.classifier.in_features,
        NUM_CLASSES
    )

    return model.to(DEVICE)


baseline = build_densenet121()

print(
    "DenseNet parameters:",
    f"{sum(p.numel() for p in baseline.parameters()):,}"
)


DenseNet parameters: 6,968,206


In [11]:
# 10. CBAM

class ChannelAttention(nn.Module):

    def __init__(
        self,
        channels,
        reduction=16
    ):
        super().__init__()

        hidden = max(
            channels // reduction,
            1
        )

        self.mlp = nn.Sequential(
            nn.Conv2d(
                channels,
                hidden,
                1,
                bias=False
            ),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                hidden,
                channels,
                1,
                bias=False
            )
        )

    def forward(self, x):

        avg = F.adaptive_avg_pool2d(
            x, 1
        )

        mx = F.adaptive_max_pool2d(
            x, 1
        )

        return torch.sigmoid(
            self.mlp(avg) +
            self.mlp(mx)
        )


class SpatialAttention(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size=7,
            padding=3,
            bias=False
        )

    def forward(self, x):

        avg = torch.mean(
            x,
            dim=1,
            keepdim=True
        )

        mx, _ = torch.max(
            x,
            dim=1,
            keepdim=True
        )

        combined = torch.cat(
            [avg, mx],
            dim=1
        )

        return torch.sigmoid(
            self.conv(combined)
        )


class CBAM(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.channel = ChannelAttention(
            channels
        )

        self.spatial = SpatialAttention()

    def forward(self, x):

        x = x * self.channel(x)
        x = x * self.spatial(x)

        return x


class DenseNet121CBAM(nn.Module):

    def __init__(self):

        super().__init__()

        base = models.densenet121(
            weights=models.DenseNet121_Weights.DEFAULT
        )

        self.features = base.features

        self.cbam = CBAM(1024)

        self.classifier = nn.Linear(
            1024,
            NUM_CLASSES
        )

    def forward(self, x):

        x = self.features(x)

        x = F.relu(
            x,
            inplace=True
        )

        x = self.cbam(x)

        x = F.adaptive_avg_pool2d(
            x, 1
        )

        x = torch.flatten(
            x,
            1
        )

        return self.classifier(x)


cbam_model = DenseNet121CBAM().to(DEVICE)

print(
    "DenseNet+CBAM parameters:",
    f"{sum(p.numel() for p in cbam_model.parameters()):,}"
)


DenseNet+CBAM parameters: 7,099,376


In [12]:
# 11. FAST TRAINING ENGINE
# Current PyTorch AMP API — avoids the previous deprecation warnings.

def amp_context():

    return torch.amp.autocast(
        device_type="cuda",
        enabled=USE_AMP
    )


def make_scaler():

    return torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )


def set_trainable(
    model,
    is_cbam=False,
    unfreeze_last_blocks=None
):

    for p in model.features.parameters():
        p.requires_grad = False

    for p in model.classifier.parameters():
        p.requires_grad = True

    if is_cbam:
        for p in model.cbam.parameters():
            p.requires_grad = True

    if unfreeze_last_blocks is not None:

        children = list(
            model.features.children()
        )

        for block in children[
            -unfreeze_last_blocks:
        ]:

            for p in block.parameters():
                p.requires_grad = True


def train_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler
):

    model.train()

    running_loss = 0.0
    total = 0

    for x, y in loader:

        x = x.to(
            DEVICE,
            non_blocking=True
        )

        y = y.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with amp_context():

            logits = model(x)

            loss = criterion(
                logits,
                y
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        scaler.step(optimizer)
        scaler.update()

        bs = x.size(0)

        running_loss += (
            loss.item() * bs
        )

        total += bs

    return (
        running_loss /
        max(total, 1)
    )


@torch.no_grad()
def get_predictions(
    model,
    loader
):

    model.eval()

    ys = []
    probs = []

    for x, y in loader:

        x = x.to(
            DEVICE,
            non_blocking=True
        )

        with amp_context():

            logits = model(x)

        probs.append(
            torch.sigmoid(
                logits
            ).float().cpu().numpy()
        )

        ys.append(
            y.numpy()
        )

    return (
        np.concatenate(ys),
        np.concatenate(probs)
    )


def mean_auc(y_true, y_prob):

    vals = []

    for i in range(NUM_CLASSES):

        try:
            vals.append(
                roc_auc_score(
                    y_true[:, i],
                    y_prob[:, i]
                )
            )

        except ValueError:
            vals.append(np.nan)

    return (
        float(np.nanmean(vals)),
        np.asarray(vals)
    )


def mean_ap(y_true, y_prob):

    vals = []

    for i in range(NUM_CLASSES):

        try:
            vals.append(
                average_precision_score(
                    y_true[:, i],
                    y_prob[:, i]
                )
            )

        except ValueError:
            vals.append(np.nan)

    return (
        float(np.nanmean(vals)),
        np.asarray(vals)
    )


def best_thresholds(
    y_true,
    y_prob
):

    grid = np.arange(
        0.10,
        0.91,
        0.05
    )

    thresholds = []

    for i in range(NUM_CLASSES):

        scores = []

        for t in grid:

            pred = (
                y_prob[:, i] >= t
            ).astype(int)

            scores.append(
                f1_score(
                    y_true[:, i],
                    pred,
                    zero_division=0
                )
            )

        thresholds.append(
            float(
                grid[
                    int(np.argmax(scores))
                ]
            )
        )

    return np.asarray(thresholds)


def evaluate(
    model,
    loader
):

    y_true, y_prob = get_predictions(
        model,
        loader
    )

    auc, aucs = mean_auc(
        y_true,
        y_prob
    )

    ap, aps = mean_ap(
        y_true,
        y_prob
    )

    thresholds = best_thresholds(
        y_true,
        y_prob
    )

    y_pred = (
        y_prob >=
        thresholds[None, :]
    ).astype(int)

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    micro_f1 = f1_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    return {
        "mean_auc": auc,
        "aucs": aucs,
        "macro_ap": ap,
        "aps": aps,
        "macro_f1": float(macro_f1),
        "micro_f1": float(micro_f1),
        "thresholds": thresholds,
        "y_true": y_true,
        "y_prob": y_prob
    }


def save_state(
    model,
    path
):

    torch.save(
        {
            "model_state_dict":
                model.state_dict(),
            "pathologies":
                PATHOLOGIES,
            "image_size":
                IMAGE_SIZE
        },
        path
    )


def train_model(
    model,
    name,
    is_cbam=False
):

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=POS_WEIGHT
    )

    scaler = make_scaler()

    best_auc = -np.inf

    best_state = copy.deepcopy(
        model.state_dict()
    )

    history = []

    # --------------------------------------------------------
    # PHASE 1 — HEAD
    # --------------------------------------------------------

    set_trainable(
        model,
        is_cbam=is_cbam,
        unfreeze_last_blocks=None
    )

    optimizer = torch.optim.AdamW(
        [
            p for p in model.parameters()
            if p.requires_grad
        ],
        lr=LR_HEAD,
        weight_decay=WEIGHT_DECAY
    )

    no_improve = 0

    for epoch in range(HEAD_EPOCHS):

        start = time.time()

        train_loss = train_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler
        )

        val = evaluate(
            model,
            val_loader
        )

        minutes = (
            time.time() - start
        ) / 60

        history.append({
            "phase": "head",
            "epoch": epoch + 1,
            "loss": train_loss,
            "val_auc": val["mean_auc"],
            "val_ap": val["macro_ap"],
            "val_f1": val["macro_f1"],
            "minutes": minutes
        })

        print(
            f"{name} | HEAD "
            f"{epoch+1}/{HEAD_EPOCHS} | "
            f"loss={train_loss:.4f} | "
            f"val_AUROC={val['mean_auc']:.4f} | "
            f"{minutes:.2f} min"
        )

        if val["mean_auc"] > best_auc:

            best_auc = val["mean_auc"]

            best_state = copy.deepcopy(
                model.state_dict()
            )

            no_improve = 0

            save_state(
                model,
                OUTPUT_DIR /
                f"{name}_best.pth"
            )

        else:

            no_improve += 1

        if no_improve >= PATIENCE:

            print(
                "Early stopping — head phase."
            )

            break

    # --------------------------------------------------------
    # PHASE 2 — FINE-TUNE LAST FEATURES
    # --------------------------------------------------------

    set_trainable(
        model,
        is_cbam=is_cbam,
        unfreeze_last_blocks=4
    )

    optimizer = torch.optim.AdamW(
        [
            p for p in model.parameters()
            if p.requires_grad
        ],
        lr=LR_FINETUNE,
        weight_decay=WEIGHT_DECAY
    )

    no_improve = 0

    for epoch in range(
        FINETUNE_EPOCHS
    ):

        start = time.time()

        train_loss = train_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler
        )

        val = evaluate(
            model,
            val_loader
        )

        minutes = (
            time.time() - start
        ) / 60

        history.append({
            "phase": "finetune",
            "epoch": epoch + 1,
            "loss": train_loss,
            "val_auc": val["mean_auc"],
            "val_ap": val["macro_ap"],
            "val_f1": val["macro_f1"],
            "minutes": minutes
        })

        print(
            f"{name} | FINE "
            f"{epoch+1}/{FINETUNE_EPOCHS} | "
            f"loss={train_loss:.4f} | "
            f"val_AUROC={val['mean_auc']:.4f} | "
            f"{minutes:.2f} min"
        )

        if val["mean_auc"] > best_auc:

            best_auc = val["mean_auc"]

            best_state = copy.deepcopy(
                model.state_dict()
            )

            no_improve = 0

            save_state(
                model,
                OUTPUT_DIR /
                f"{name}_best.pth"
            )

        else:

            no_improve += 1

        if no_improve >= PATIENCE:

            print(
                "Early stopping — fine-tune phase."
            )

            break

    model.load_state_dict(
        best_state
    )

    history_df = pd.DataFrame(
        history
    )

    history_df.to_csv(
        OUTPUT_DIR /
        f"{name}_history.csv",
        index=False
    )

    print(
        f"{name} best validation AUROC: "
        f"{best_auc:.4f}"
    )

    return model, history_df


In [13]:
# 12. TRAIN DENSENET-121

baseline, baseline_history = train_model(
    baseline,
    name="DenseNet121",
    is_cbam=False
)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


DenseNet121 | HEAD 1/2 | loss=1.3513 | val_AUROC=0.6806 | 5.78 min
DenseNet121 | HEAD 2/2 | loss=1.2706 | val_AUROC=0.7122 | 5.05 min
DenseNet121 | FINE 1/5 | loss=1.2079 | val_AUROC=0.7506 | 5.28 min
DenseNet121 | FINE 2/5 | loss=1.1190 | val_AUROC=0.7620 | 5.18 min
DenseNet121 | FINE 3/5 | loss=1.0575 | val_AUROC=0.7688 | 5.29 min
DenseNet121 | FINE 4/5 | loss=1.0119 | val_AUROC=0.7738 | 5.08 min
DenseNet121 | FINE 5/5 | loss=0.9658 | val_AUROC=0.7775 | 5.14 min
DenseNet121 best validation AUROC: 0.7775


In [14]:
# 13. TRAIN DENSENET-121 + CBAM

cbam_model, cbam_history = train_model(
    cbam_model,
    name="DenseNet121_CBAM",
    is_cbam=True
)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


DenseNet121_CBAM | HEAD 1/2 | loss=1.3374 | val_AUROC=0.7000 | 5.03 min
DenseNet121_CBAM | HEAD 2/2 | loss=1.2705 | val_AUROC=0.7259 | 5.30 min
DenseNet121_CBAM | FINE 1/5 | loss=1.1671 | val_AUROC=0.7569 | 5.59 min
DenseNet121_CBAM | FINE 2/5 | loss=1.1043 | val_AUROC=0.7675 | 5.49 min
DenseNet121_CBAM | FINE 3/5 | loss=1.0629 | val_AUROC=0.7739 | 5.54 min
DenseNet121_CBAM | FINE 4/5 | loss=1.0212 | val_AUROC=0.7774 | 5.10 min
DenseNet121_CBAM | FINE 5/5 | loss=0.9773 | val_AUROC=0.7805 | 5.23 min
DenseNet121_CBAM best validation AUROC: 0.7805


In [15]:
# 14. FINAL DEVELOPMENT-TEST COMPARISON
# Thresholds are selected on validation data, then applied unchanged to test data.

print("Selecting thresholds from validation set...")

baseline_val = evaluate(baseline, val_loader)
cbam_val = evaluate(cbam_model, val_loader)

baseline_thresholds = baseline_val["thresholds"]
cbam_thresholds = cbam_val["thresholds"]


def evaluate_with_thresholds(model, loader, thresholds):
    y_true, y_prob = get_predictions(model, loader)

    auc, aucs = mean_auc(y_true, y_prob)
    ap, aps = mean_ap(y_true, y_prob)

    y_pred = (
        y_prob >= thresholds[None, :]
    ).astype(int)

    macro_f1 = f1_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    )

    micro_f1 = f1_score(
        y_true, y_pred,
        average="micro",
        zero_division=0
    )

    return {
        "mean_auc": auc,
        "aucs": aucs,
        "macro_ap": ap,
        "aps": aps,
        "macro_f1": float(macro_f1),
        "micro_f1": float(micro_f1),
        "thresholds": thresholds,
        "y_true": y_true,
        "y_prob": y_prob
    }


print("Evaluating DenseNet-121 on development test...")

baseline_test = evaluate_with_thresholds(
    baseline,
    test_loader,
    baseline_thresholds
)

print("Evaluating DenseNet-121 + CBAM on development test...")

cbam_test = evaluate_with_thresholds(
    cbam_model,
    test_loader,
    cbam_thresholds
)

comparison = pd.DataFrame([
    {
        "Model": "DenseNet-121",
        "Mean AUROC": baseline_test["mean_auc"],
        "Macro AP": baseline_test["macro_ap"],
        "Macro F1": baseline_test["macro_f1"],
        "Micro F1": baseline_test["micro_f1"],
    },
    {
        "Model": "DenseNet-121 + CBAM",
        "Mean AUROC": cbam_test["mean_auc"],
        "Macro AP": cbam_test["macro_ap"],
        "Macro F1": cbam_test["macro_f1"],
        "Micro F1": cbam_test["micro_f1"],
    }
])

display(
    comparison.style.format({
        "Mean AUROC": "{:.4f}",
        "Macro AP": "{:.4f}",
        "Macro F1": "{:.4f}",
        "Micro F1": "{:.4f}",
    })
)

comparison.to_csv(
    OUTPUT_DIR / "model_comparison.csv",
    index=False
)

print()
print(
    "CBAM Mean AUROC gain:",
    f"{cbam_test['mean_auc'] - baseline_test['mean_auc']:+.4f}"
)


Selecting thresholds from validation set...
Evaluating DenseNet-121 on development test...
Evaluating DenseNet-121 + CBAM on development test...


,Model,Mean AUROC,Macro AP,Macro F1,Micro F1
0,DenseNet-121,0.7255,0.1615,0.2077,0.2988
1,DenseNet-121 + CBAM,0.7144,0.1552,0.2052,0.2859



CBAM Mean AUROC gain: -0.0111


In [16]:
# 15. PER-DISEASE AUROC

per_disease = pd.DataFrame({
    "Disease": PATHOLOGIES,
    "DenseNet121_AUROC":
        baseline_test["aucs"],
    "CBAM_AUROC":
        cbam_test["aucs"]
})

per_disease["Delta_AUROC"] = (
    per_disease["CBAM_AUROC"] -
    per_disease["DenseNet121_AUROC"]
)

display(
    per_disease.style.format({
        "DenseNet121_AUROC": "{:.4f}",
        "CBAM_AUROC": "{:.4f}",
        "Delta_AUROC": "{:+.4f}"
    })
)

per_disease.to_csv(
    OUTPUT_DIR /
    "per_disease_auroc.csv",
    index=False
)


,Disease,DenseNet121_AUROC,CBAM_AUROC,Delta_AUROC
0,Atelectasis,0.7002,0.6924,-0.0078
1,Cardiomegaly,0.7595,0.7323,-0.0272
2,Effusion,0.7393,0.7245,-0.0148
3,Infiltration,0.6730,0.6731,+0.0002
4,Mass,0.6356,0.6274,-0.0082
5,Nodule,0.6765,0.6786,+0.0021
6,Pneumonia,0.6309,0.6052,-0.0257
7,Pneumothorax,0.7616,0.7596,-0.0019
8,Consolidation,0.7008,0.6951,-0.0057
9,Edema,0.7723,0.7679,-0.0044


In [17]:
# 16. SAVE FINAL FAST-EXPERIMENT MODELS

baseline_path = (
    OUTPUT_DIR /
    "MediSense_Chest_DenseNet121_fast.pth"
)

cbam_path = (
    OUTPUT_DIR /
    "MediSense_Chest_DenseNet121_CBAM_fast.pth"
)

save_state(
    baseline,
    baseline_path
)

save_state(
    cbam_model,
    cbam_path
)

with open(
    OUTPUT_DIR /
    "experiment_metadata.json",
    "w"
) as f:

    json.dump(
        {
            "environment":
                "Kaggle GPU",
            "dataset":
                "NIH ChestX-ray14",
            "purpose":
                "MediSense model selection",
            "num_classes":
                NUM_CLASSES,
            "pathologies":
                PATHOLOGIES,
            "train_images":
                len(train_run_df),
            "validation_images":
                len(val_run_df),
            "test_images":
                len(test_run_df),
            "image_size":
                IMAGE_SIZE,
            "batch_size":
                BATCH_SIZE,
            "baseline_mean_auc":
                baseline_test["mean_auc"],
            "cbam_mean_auc":
                cbam_test["mean_auc"],
            "cbam_auc_gain":
                (
                    cbam_test["mean_auc"] -
                    baseline_test["mean_auc"]
                )
        },
        f,
        indent=2
    )

print()
print("OUTPUT DIRECTORY:")
print(OUTPUT_DIR)
print()
print("DenseNet:", baseline_path)
print("CBAM:", cbam_path)



OUTPUT DIRECTORY:
/kaggle/working/medisense_chest_densenet_cbam_fast

DenseNet: /kaggle/working/medisense_chest_densenet_cbam_fast/MediSense_Chest_DenseNet121_fast.pth
CBAM: /kaggle/working/medisense_chest_densenet_cbam_fast/MediSense_Chest_DenseNet121_CBAM_fast.pth


In [23]:
# 15A. ACCURACY + PRECISION + RECALL

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score
)

def calculate_final_metrics(result):
    y_true = result["y_true"]
    y_prob = result["y_prob"]
    thresholds = result["thresholds"]

    # Apply validation-derived thresholds
    y_pred = (
        y_prob >= thresholds[None, :]
    ).astype(int)

    # Exact-match / subset accuracy:
    # ALL 14 labels must be correct for an image
    exact_match_accuracy = accuracy_score(
        y_true,
        y_pred
    )

    # Label-wise accuracy:
    # Percentage of all individual disease labels classified correctly
    label_wise_accuracy = np.mean(
        y_true == y_pred
    )

    macro_precision = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    return {
        "Exact Match Accuracy": exact_match_accuracy,
        "Label-wise Accuracy": label_wise_accuracy,
        "Macro Precision": macro_precision,
        "Macro Recall": macro_recall,
        "Macro F1": result["macro_f1"],
        "Micro F1": result["micro_f1"],
        "Mean AUROC": result["mean_auc"],
        "Macro AP": result["macro_ap"]
    }


# Calculate metrics
baseline_final = calculate_final_metrics(
    baseline_test
)

cbam_final = calculate_final_metrics(
    cbam_test
)


# Create comparison table
final_comparison = pd.DataFrame([
    {
        "Model": "DenseNet-121",
        **baseline_final
    },
    {
        "Model": "DenseNet-121 + CBAM",
        **cbam_final
    }
])

display(
    final_comparison.style.format({
        "Exact Match Accuracy": "{:.2%}",
        "Label-wise Accuracy": "{:.2%}",
        "Macro Precision": "{:.2%}",
        "Macro Recall": "{:.2%}",
        "Macro F1": "{:.2%}",
        "Micro F1": "{:.2%}",
        "Mean AUROC": "{:.2%}",
        "Macro AP": "{:.2%}"
    })
)

,Model,Exact Match Accuracy,Label-wise Accuracy,Macro Precision,Macro Recall,Macro F1,Micro F1,Mean AUROC,Macro AP
0,DenseNet-121,18.92%,81.94%,17.44%,35.24%,20.77%,29.88%,72.55%,16.15%
1,DenseNet-121 + CBAM,16.66%,80.61%,15.80%,35.61%,20.52%,28.59%,71.44%,15.52%


# 17. WHAT TO DO AFTER THIS RUN

### Step 1
Run the entire notebook on Kaggle GPU.

### Step 2
Do **not** interrupt because of normal training output.

### Step 3
At the end, send the **model comparison table** to me.

We will decide:

**DenseNet-121 vs DenseNet-121 + CBAM**

based primarily on Mean AUROC and then inspect the 14 individual disease AUROCs.

### Important

This fast run is for **MediSense model selection and efficient experimentation**.

The 5,000-image development test subset should not be presented as the definitive performance of the complete NIH ChestX-ray14 test set. If the CBAM model clearly wins, we can then run a final evaluation of the selected model on the official test set before making a research claim.

The final MediSense model should be the `.pth` produced after the selected architecture has been validated.
